# Tickets de soporte: limpieza, tokenización y features TF-IDF

1. Descarga y carga del dataset
2. Limpieza (body vacío y tickets en alemán)
3. Preprocesado y tokenización (oraciones, normalización, tokens) + split train/test
4. Vocabulario (clase `Vocabulary`, con token `<UNK>` y corte por frecuencia)
5. Representación TF-IDF
6. Top palabras global y por cola
7. Corpus de texto para Word2Vec (incluye `answer`, solo train)
8. Word embeddings preentrenados (Word2Vec Google News) y análisis de palabras sin vector (OOV)
9. Word2Vec propio: entrenamiento, comparación de cobertura y decisión
10. Exportación estructurada (JSONL + metadatos) para reutilizar el corpus

In [1]:
import json
import os
import re
import string
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset, load_from_disk
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from spacy.lang.en.stop_words import STOP_WORDS as STOP_EN

C:\Users\gonzalo.iglesias\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración y carga de datos

In [2]:
PATH = (os.path.join("..","data", "tickets"))
CSV_PATH = os.path.join(PATH, "raw/dataraw.csv")
PROCESSED_PATH = os.path.join(PATH, "dataprocessed/dataprocessed.csv")
OUTDIR = Path(PATH) / "features"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [3]:
if not os.path.exists(os.path.join(PATH, "dataset_dict.json")):
    load_dataset("Tobi-Bueck/customer-support-tickets").save_to_disk(PATH)

if not os.path.exists(CSV_PATH):
    ds = load_from_disk(PATH)
    # SOLO TIENE TRAIN, POR LO QUE TENEMOS QUE HACER NOSOTROS EL SPLIT
    df = ds["train"].to_pandas()
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    df.to_csv(CSV_PATH, index=False)
else:
    df = pd.read_csv(CSV_PATH)

## 2. Limpieza de datos

In [4]:
def limpieza_datos(df: pd.DataFrame, output_path: str = PROCESSED_PATH) -> pd.DataFrame:
    """Elimina las filas con 'body' vacío (NaN), y los tickets escritos en alemán y guarda el resultado en CSV."""
    n_antes = len(df)
    df_limpio = df.dropna(subset=["body"]).reset_index(drop=True)
    df_limpio = df_limpio.drop(df_limpio[df_limpio["language"] == "de"].index)
    print(f"Filas eliminadas: {n_antes - len(df_limpio)} (quedan {len(df_limpio)})")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_limpio.to_csv(output_path, index=False)
    return df_limpio

In [5]:
df = limpieza_datos(df)
df.head()

Filas eliminadas: 33505 (quedan 28260)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN


> Tenemos también las respuestas (`answer`). Para la primera tarea (clasificar el ticket) esa columna **no se usa como feature**: no existe respuesta en el momento de la clasificación. Por eso `answer` nunca entra en `tokens`, ni en el vocabulario, ni en TF-IDF. Sí se procesa (con las mismas funciones) y se usa, **solo con los tickets de train**, como texto adicional para entrenar Word2Vec (ver sección 7).
>
> Lo ideal sería predecir: **queue** (a quién va dirigido), **priority** (importancia) y **tags**.

## 3. Preprocesado y tokenización

Siguiendo el formato de texto que se usa en la práctica de clasificación con Perceptron (Yelp): **texto en minúsculas, con la puntuación separada por espacios**, de forma que `text.split(" ")` da directamente la lista de tokens y la puntuación se descarta con `token not in string.punctuation`. Aquí ese formato lo generamos nosotros a partir del texto crudo, en tres pasos:

1. **División en oraciones**: se parte el texto (`subject` + `body`, y por separado `answer`) en saltos de línea y tras `.`, `!` o `?`. Se hace sobre el texto original, antes de pasar a minúsculas.
2. **Normalización**: URLs, emails y números se sustituyen por los marcadores `<url>`, `<email>` y `<num>` (reducen vocabulario y evitan tokens casi únicos). Después se pasa a minúsculas.
3. **Tokenización** con una expresión regular que mantiene como un solo token:
   - los marcadores entre `<...>` (también los que ya trae el dataset, como `<name>`),
   - las palabras con apóstrofo (`don't`),
   - cada signo de puntuación suelto.

Columnas resultantes:

| columna | contenido |
|---|---|
| `sentences` | lista de oraciones (JSON) |
| `n_sentences`, `n_tokens` | recuentos |
| `text_tok` | tokens con puntuación, separados por espacios (formato Yelp) |
| `tokens` | tokens sin puntuación, separados por espacios (entrada del vocabulario y de TF-IDF) |
| `answer_sentences` | oraciones de la respuesta (JSON); solo para el corpus de Word2Vec |
| `answer_tokens`, `n_tokens_answer` | tokens sin puntuación de la respuesta y su recuento; **no** se usan como feature |
| `split` | `train` / `test` |

**Saltos de línea literales.** En el dataset los saltos de línea de `body` y `answer` vienen como los dos caracteres `\n` (barra invertida + n), no como saltos reales. Sin corregirlo, el divisor de oraciones no corta en ellos y el tokenizador genera tokens basura (`\n\nI` pasaba a `n` + `ni`, `\nThank` a `nthank`). La función `fix_newlines` los convierte en saltos reales antes de dividir en oraciones, tanto en el ticket como en la respuesta.

No se lematiza ni se eliminan stopwords en este paso: las stopwords se filtran más adelante en TF-IDF y se conservan para los embeddings.

In [6]:
PUNCT = set(string.punctuation)

URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)*\b")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
TOKEN_RE = re.compile(r"<[a-z_]+>|\w+(?:'\w+)*|[^\w\s]")


def split_sentences(text: str) -> list[str]:
    """Divide un texto en oraciones (saltos de línea y tras . ! ?)."""
    return [s.strip() for s in SENT_SPLIT_RE.split(text) if s and s.strip()]


def normalize(sentence: str) -> str:
    """Sustituye URLs, emails y números por marcadores y pasa a minúsculas."""
    s = URL_RE.sub(" <url> ", sentence)
    s = EMAIL_RE.sub(" <email> ", s)
    s = NUM_RE.sub(" <num> ", s)
    return s.lower()


def tokenize(sentence: str) -> list[str]:
    """Tokeniza una oración: palabras, marcadores <...> y puntuación suelta."""
    return TOKEN_RE.findall(normalize(sentence))


def fix_newlines(text: str) -> str:
    """El dataset guarda los saltos de línea como los dos caracteres '\\n'; los pasamos a saltos reales."""
    return text.replace("\\n", "\n")

In [7]:
df = df.reset_index(drop=True)

# Un texto por ticket: asunto + cuerpo (para clasificar NO se usa 'answer': no existe en ese momento)
raw_text = (df["subject"].fillna("") + "\n" + df["body"]).map(fix_newlines).str.strip()

sentences = [split_sentences(t) for t in raw_text]
sent_tokens = [[tokenize(s) for s in sents] for sents in sentences]

df["sentences"] = [json.dumps(s, ensure_ascii=False) for s in sentences]
df["n_sentences"] = [len(s) for s in sentences]
df["text_tok"] = [" ".join(t for sent in st for t in sent) for st in sent_tokens]
df["tokens"] = [" ".join(t for sent in st for t in sent if t not in PUNCT) for st in sent_tokens]
df["n_tokens"] = df["tokens"].str.split().str.len()

# La respuesta (answer) se procesa igual, pero en columnas aparte: solo servirá como texto extra para Word2Vec (sección 7)
raw_answer = df["answer"].fillna("").map(fix_newlines).str.strip()
answer_sentences = [split_sentences(t) for t in raw_answer]
answer_sent_tokens = [[tokenize(s) for s in sents] for sents in answer_sentences]

df["answer_sentences"] = [json.dumps(s, ensure_ascii=False) for s in answer_sentences]
df["answer_tokens"] = [" ".join(t for sent in st for t in sent if t not in PUNCT) for st in answer_sent_tokens]
df["n_tokens_answer"] = df["answer_tokens"].str.split().str.len()

# Ejemplo de la transformación
i = 0
print("RAW        :", raw_text[i][:300].replace("\n", " | "))
print("ORACIONES  :", sentences[i][:3])
print("TEXT_TOK   :", df.loc[i, "text_tok"][:300])
print("TOKENS     :", df.loc[i, "tokens"][:300])
print("ANSWER_TOK :", df.loc[i, "answer_tokens"][:300])
print()
print(df[["n_sentences", "n_tokens", "n_tokens_answer"]].describe().round(1))

RAW        : Account Disruption | Dear Customer Support Team, |  | I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in mul
ORACIONES  : ['Account Disruption', 'Dear Customer Support Team,', 'I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline.']
TEXT_TOK   : account disruption dear customer support team , i am writing to report a significant problem with the centralized account management portal , which currently appears to be offline . this outage is blocking access to account settings , leading to substantial inconvenience . i have attempted to log in
TOKENS     : account disruption dear customer support team i am writing to report a significant problem with the centralized account management portal which current

In [8]:
# El dataset solo trae 'train', así que el split lo hacemos aquí (antes de construir el vocabulario)
idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=df["queue"],  # quita esta línea si alguna cola tiene muy pocos tickets
)
df["split"] = "train"
df.loc[idx_test, "split"] = "test"
print(df["split"].value_counts())

# Guardamos el CSV procesado (ahora con tokens y split)
df.to_csv(PROCESSED_PATH, index=False)

split
train    22608
test      5652
Name: count, dtype: int64


## 4. Vocabulario

Clase `Vocabulary` basada en la de la práctica (Rao & McMahan, 2019): mapea token ↔ índice y reserva un token `<UNK>` para las palabras desconocidas. Se construye **solo con el train** (evita fuga de información) y con un **corte por frecuencia**: los tokens con menos apariciones que `MIN_FREQ` se representan como `<UNK>`. Se añade `to_serializable` / `from_serializable` para guardarlo en JSON y reutilizarlo.

In [9]:
# Basado en Rao, D., & McMahan, B. (2019). Natural language processing with PyTorch.
class Vocabulary(object):
    """Clase para procesar texto y extraer el vocabulario existente para su posterior mapeo."""

    def __init__(self, token_to_idx=None, add_unk=True, unk_token="<UNK>"):
        """
        Args: token_to_idx (dict): mapa token -> índice preexistente.
            : add_unk (bool): si se añade el token UNK.
            : unk_token (str): el token UNK.
        """
        if token_to_idx is None:
            token_to_idx = {}
        self._token_to_idx = token_to_idx
        self._idx_to_token = {idx: token for token, idx in self._token_to_idx.items()}
        self._add_unk = add_unk
        self._unk_token = unk_token
        self.unk_index = -1
        if add_unk:
            self.unk_index = self.add_token(unk_token)

    def to_serializable(self):
        return {"token_to_idx": self._token_to_idx,
                "add_unk": self._add_unk,
                "unk_token": self._unk_token}

    @classmethod
    def from_serializable(cls, contents):
        return cls(**contents)

    def add_token(self, token):
        """Añade un token (si no existe) y devuelve su índice."""
        if token in self._token_to_idx:
            index = self._token_to_idx[token]
        else:
            index = len(self._token_to_idx)
            self._token_to_idx[token] = index
            self._idx_to_token[index] = token
        return index

    def add_many(self, tokens):
        return [self.add_token(token) for token in tokens]

    def lookup_token(self, token):
        """Índice del token; si no existe devuelve el de <UNK>."""
        if self.unk_index >= 0:
            return self._token_to_idx.get(token, self.unk_index)
        return self._token_to_idx[token]

    def lookup_index(self, index):
        if index not in self._idx_to_token:
            raise KeyError("the index (%d) is not in the Vocabulary" % index)
        return self._idx_to_token[index]

    def __str__(self):
        return "<Vocabulary (size=%d)>" % len(self)

    def __len__(self):
        return len(self._token_to_idx)

In [10]:
MIN_FREQ = 2  # corte por frecuencia: tokens con menos apariciones en train -> <UNK>

train_tokens = [t for toks in df.loc[df["split"] == "train", "tokens"] for t in toks.split()]
word_counts = Counter(train_tokens)

vocab = Vocabulary(add_unk=True, unk_token="<UNK>")
for word, count in word_counts.most_common():  # ordenado por frecuencia
    if count >= MIN_FREQ:
        vocab.add_token(word)

print(vocab)
print(f"Tipos distintos en train: {len(word_counts)} -> en el vocabulario: {len(vocab)}")

# Cobertura sobre test: qué fracción de tokens cae en <UNK>
test_tokens = [t for toks in df.loc[df["split"] == "test", "tokens"] for t in toks.split()]
n_unk = sum(vocab.lookup_token(t) == vocab.unk_index for t in test_tokens)
print(f"Tokens de test desconocidos (<UNK>): {n_unk}/{len(test_tokens)} ({n_unk / len(test_tokens):.2%})")

# Guardamos el vocabulario para reutilizarlo
with open(OUTDIR / "vocab.json", "w", encoding="utf-8") as fp:
    json.dump(vocab.to_serializable(), fp, ensure_ascii=False)

<Vocabulary (size=4897)>
Tipos distintos en train: 5923 -> en el vocabulario: 4897
Tokens de test desconocidos (<UNK>): 551/345339 (0.16%)


## 5. Features TF-IDF

In [11]:
# Usamos los tokens ya procesados (sin puntuación, en minúsculas): el vectorizador
# solo hace split por espacios, no vuelve a tokenizar ni a pasar a minúsculas.
data = df
corpus = data["tokens"].fillna("").to_numpy()
idx_train = np.where(data["split"] == "train")[0]
idx_test = np.where(data["split"] == "test")[0]

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    stop_words=sorted(STOP_EN),  # solo inglés: los tickets en alemán ya se eliminaron
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

# Ajustar SOLO con train para evitar fuga de información
X_train = vectorizer.fit_transform(corpus[idx_train])
X_test = vectorizer.transform(corpus[idx_test])

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Vocabulario: {len(vectorizer.get_feature_names_out())}")

Train: (22608, 71191), Test: (5652, 71191)
Vocabulario: 71191


In [12]:
# Guardar para reutilizar
sparse.save_npz(OUTDIR / "X_train.npz", X_train)
sparse.save_npz(OUTDIR / "X_test.npz", X_test)
np.save(OUTDIR / "idx_train.npy", idx_train)  # para recuperar queue/priority/tags
np.save(OUTDIR / "idx_test.npy", idx_test)
joblib.dump(vectorizer, OUTDIR / "tfidf_vectorizer.joblib")

['..\\data\\tickets\\features\\tfidf_vectorizer.joblib']

## 6. Top palabras

In [13]:
TOP_N = 5
feature_names = vectorizer.get_feature_names_out()

# 1) Top global: peso TF-IDF medio por término en train
mean_tfidf = np.asarray(X_train.mean(axis=0)).ravel()
top_global = np.argsort(mean_tfidf)[::-1][:TOP_N]

print(f"Top {TOP_N} palabras (TF-IDF medio, global):")
for i in top_global:
    print(f"  {feature_names[i]:<30} {mean_tfidf[i]:.4f}")

Top 5 palabras (TF-IDF medio, global):
  data                           0.0326
  issue                          0.0195
  security                       0.0190
  medical                        0.0178
  analytics                      0.0175


In [14]:
# 2) Top por cola (queue): TF-IDF medio solo con los tickets de esa cola
y_train = data["queue"].iloc[idx_train].to_numpy()
top_por_cola = {}

for cola in np.unique(y_train):
    mask = y_train == cola
    media_cola = np.asarray(X_train[mask].mean(axis=0)).ravel()
    top_idx = np.argsort(media_cola)[::-1][:TOP_N]
    top_por_cola[cola] = [(feature_names[i], media_cola[i]) for i in top_idx]

    print(f"\nTop {TOP_N} palabras para la cola '{cola}':")
    for palabra, peso in top_por_cola[cola]:
        print(f"  {palabra:<30} {peso:.4f}")


Top 5 palabras para la cola 'Billing and Payments':
  billing                        0.0735
  payment                        0.0439
  information                    0.0247
  data                           0.0224
  services                       0.0221

Top 5 palabras para la cola 'Customer Service':
  data                           0.0359
  investment                     0.0208
  analytics                      0.0196
  security                       0.0195
  information                    0.0195

Top 5 palabras para la cola 'General Inquiry':
  data                           0.0399
  investment                     0.0250
  engagement                     0.0240
  digital                        0.0232
  strategies                     0.0212

Top 5 palabras para la cola 'Human Resources':
  data                           0.0341
  software                       0.0226
  employee                       0.0224
  security                       0.0203
  issue                          0.0182

T

## 7. Corpus de texto para Word2Vec (incluye `answer`)

Para entrenar nuestro propio Word2Vec se puede añadir `answer` como texto extra, porque más texto del mismo dominio mejora los vectores. Las reglas que seguimos y su justificación:

- **Solo tickets de train.** Tanto `subject` + `body` como `answer` se toman únicamente de las filas con `split == "train"`. Así ni el texto de test ni sus respuestas influyen en los embeddings (evita fuga de información), igual que ya hacemos con el vocabulario y con TF-IDF.
- **`answer` no es una feature.** Word2Vec no usa etiquetas (es no supervisado) y aquí solo aporta contexto de palabras. La respuesta no se usa para clasificar, y `tokens`, `Vocabulary` y TF-IDF se siguen construyendo únicamente con `subject` + `body`.
- **La cobertura de OOV se mide solo sobre `subject` + `body`.** Es el texto que verá el clasificador en producción; medirla también sobre `answer` sobreestimaría cuántas palabras quedan cubiertas.
- **Por qué merece la pena.** Las respuestas del equipo de soporte comparten dominio y léxico con los tickets (facturación, cuentas, integraciones...), así que aumentan el número de ocurrencias de palabras que en los tickets aparecen pocas veces.
- **Limitaciones.** Las respuestas están redactadas por otro "hablante" (registro más formal y plantillas repetidas), y el dataset es sintético con muchos textos casi idénticos. Si los vectores salen sesgados hacia ese estilo, se puede entrenar también una versión solo con tickets y comparar.

Salida: `features/w2v_corpus_train.txt`, una oración por línea con los tokens separados por espacios (sin puntuación, mismo formato que `tokens`). Se carga directamente con `gensim.models.word2vec.LineSentence`.

In [15]:
W2V_PATH = OUTDIR / "w2v_corpus_train.txt"
train_df = df[df["split"] == "train"]


def sentences_to_token_lists(json_sentences) -> list[list[str]]:
    """Columna de oraciones (JSON) -> lista de oraciones, cada una como lista de tokens sin puntuación."""
    out = []
    for js in json_sentences:
        for s in json.loads(js):
            toks = [t for t in tokenize(s) if t not in PUNCT]
            if toks:
                out.append(toks)
    return out


w2v_ticket = sentences_to_token_lists(train_df["sentences"])         # subject + body (train)
w2v_answer = sentences_to_token_lists(train_df["answer_sentences"])  # answer (train)
w2v_corpus = w2v_ticket + w2v_answer

with open(W2V_PATH, "w", encoding="utf-8") as fp:
    for toks in w2v_corpus:
        fp.write(" ".join(toks) + "\n")

n_tok_t = sum(map(len, w2v_ticket))
n_tok_a = sum(map(len, w2v_answer))
print(f"Oraciones: {len(w2v_ticket)} de tickets + {len(w2v_answer)} de respuestas = {len(w2v_corpus)}")
print(f"Tokens   : {n_tok_t} de tickets + {n_tok_a} de respuestas = {n_tok_t + n_tok_a} (las respuestas aportan {n_tok_a / (n_tok_t + n_tok_a):.1%})")
print(f"Guardado en {W2V_PATH}")

# Uso posterior:
# from gensim.models.word2vec import LineSentence, Word2Vec
# w2v = Word2Vec(LineSentence(str(W2V_PATH)), vector_size=100, window=5, min_count=2, workers=4, seed=42)

Oraciones: 109354 de tickets + 85770 de respuestas = 195124
Tokens   : 1372877 de tickets + 1323319 de respuestas = 2696196 (las respuestas aportan 49.1%)
Guardado en ..\data\tickets\features\w2v_corpus_train.txt


## 8. Word embeddings preentrenados (Word2Vec Google News) y análisis de OOV

Representamos cada palabra del vocabulario con **Word2Vec preentrenado de Google News (300 dimensiones)** (`word2vec-google-news-300`, vía `gensim.downloader`): 3 millones de palabras y expresiones entrenadas con ~100 000 millones de palabras de noticias. Es el mismo algoritmo que veremos entrenar en la sección 9, pero con un corpus general mucho mayor.

**Particularidades de este modelo frente a nuestra tokenización**

1. **Distingue mayúsculas y minúsculas**, y nuestros tokens están todos en minúsculas. Fallback en `get_pre_vec`: se prueba `palabra`, `Palabra` y `PALABRA` (hay términos que solo existen capitalizados, como nombres de producto o de empresa).
2. **No incluye algunas stopwords** (`a`, `and`, `of`, `to`...). Inflarían artificialmente el % de OOV, así que la cobertura se informa **con y sin stopwords**. Sin stopwords es la cifra relevante para los vectores de ticket, donde se descartan.
3. **Marcadores** (`<name>`, `<num>`, `<url>`, `<email>`...): los generamos nosotros, no existen en el modelo. Se **cuentan aparte** y no entran en el % de OOV. En la matriz de embeddings reciben un vector aleatorio propio (son frecuentes e informativos).
4. **Posesivos** (`customer's`): si no están, se usa el vector de la palabra base (`customer`).
5. **`<UNK>`**: vector de ceros.
6. **Memoria:** el modelo completo ocupa ~3,6 GB en RAM. Si el equipo va justo, `PRETRAINED_LIMIT` carga solo las N palabras más frecuentes (el fichero está ordenado por frecuencia); la cobertura medida será ligeramente más pesimista.

**Qué se calcula**

- Cobertura por **tipos** (palabras distintas) y por **tokens** (apariciones): un OOV muy raro pesa poco en tokens aunque cuente en tipos.
- Se mide sobre el vocabulario de train y sobre test (lo que realmente verá el modelo).
- Lista de los OOV más frecuentes y su clasificación aproximada.
- Matriz de embeddings alineada con `Vocabulary` (`emb_matrix_pretrained.npy`) y un vector por ticket (media de los vectores de sus palabras sin stopwords) para usarlo como feature.

In [16]:
import gensim.downloader as api
from gensim.models import KeyedVectors, Word2Vec
from gensim.models.word2vec import LineSentence

EMB_NAME = "word2vec-google-news-300"  # Word2Vec preentrenado, 3M palabras, 300 dimensiones (~1,6 GB de descarga)
PRETRAINED_LIMIT = None                # p. ej. 1_000_000 para cargar solo las más frecuentes si falta RAM (~3,6 GB completo)

w2v_path = api.load(EMB_NAME, return_path=True)  # descarga la primera vez y deja el fichero en caché
w2v_pre = KeyedVectors.load_word2vec_format(w2v_path, binary=True, limit=PRETRAINED_LIMIT)
EMB_DIM = w2v_pre.vector_size
print(f"{EMB_NAME}: {len(w2v_pre.key_to_index):,} palabras, {EMB_DIM} dimensiones")

word2vec-google-news-300: 3,000,000 palabras, 300 dimensiones


In [17]:
MARKER_RE = re.compile(r"^<[a-z_]+>$")


def is_marker(tok: str) -> bool:
    """True para los marcadores <name>, <num>, <url>, <email>... y <UNK>."""
    return bool(MARKER_RE.match(tok)) or tok == "<UNK>"


_pre_cache = {}


def get_pre_vec(tok: str):
    """Vector del Word2Vec preentrenado, o None si no se puede representar.
    El modelo distingue mayúsculas: se prueba minúscula, Capitalizada y MAYÚSCULAS.
    Para posesivos (customer's) se usa la palabra base si el token entero no existe."""
    if tok in _pre_cache:
        return _pre_cache[tok]
    vec = None
    candidates = [tok, tok.capitalize(), tok.upper()]
    if tok.endswith("'s") and len(tok) > 2:
        base = tok[:-2]
        candidates += [base, base.capitalize()]
    for cand in candidates:
        if cand in w2v_pre.key_to_index:
            vec = w2v_pre[cand]
            break
    _pre_cache[tok] = vec
    return vec


def coverage_report(counter: Counter, has_vec, name: str, min_count: int = 1, skip_stop: bool = False) -> dict:
    """Cobertura por tipos y por tokens, excluyendo marcadores (se informan aparte).
    skip_stop=True excluye además las stopwords (el modelo preentrenado no incluye varias, como 'a', 'and', 'of', 'to')."""
    words = {w: c for w, c in counter.items()
             if not is_marker(w) and c >= min_count and not (skip_stop and w in STOP_EN)}
    n_types, n_tokens = len(words), sum(words.values())
    cov_types = sum(has_vec(w) for w in words)
    cov_tokens = sum(c for w, c in words.items() if has_vec(w))
    return {
        "conjunto": name,
        "tipos": n_types, "tipos_sin_vector": n_types - cov_types,
        "% tipos OOV": round(100 * (n_types - cov_types) / n_types, 2),
        "tokens": n_tokens, "tokens_sin_vector": n_tokens - cov_tokens,
        "% tokens OOV": round(100 * (n_tokens - cov_tokens) / n_tokens, 2),
    }


has_pre = lambda w: get_pre_vec(w) is not None
test_counts = Counter(test_tokens)  # `test_tokens` viene de la sección 4

cov_pre = pd.DataFrame([
    coverage_report(word_counts, has_pre, "train (todos los tipos)"),
    coverage_report(word_counts, has_pre, f"train (vocabulario, freq >= {MIN_FREQ})", min_count=MIN_FREQ),
    coverage_report(test_counts, has_pre, "test"),
    coverage_report(test_counts, has_pre, "test (sin stopwords)", skip_stop=True),
])
display(cov_pre) if "display" in globals() else print(cov_pre.to_string(index=False))

# Marcadores: no son OOV, se cuentan aparte
total_train = sum(word_counts.values())
print("\nMarcadores en train (no entran en el % de OOV):")
for w, c in sorted(((w, c) for w, c in word_counts.items() if is_marker(w)), key=lambda x: -x[1]):
    print(f"  {w:<12} {c:>8}  ({c / total_train:.2%} de los tokens de train)")

                      conjunto  tipos  tipos_sin_vector  % tipos OOV  tokens  tokens_sin_vector  % tokens OOV
       train (todos los tipos)   5915               204         3.45 1367680               4715          0.34
train (vocabulario, freq >= 2)   4890               149         3.05 1366655               4660          0.34
                          test   4411               134         3.04  344092               1129          0.33
          test (sin stopwords)   4199               134         3.19  197006               1129          0.57

Marcadores en train (no entran en el % de OOV):
  <num>            2541  (0.19% de los tokens de train)
  <br>             2303  (0.17% de los tokens de train)
  <tel_num>         267  (0.02% de los tokens de train)
  <acc_num>          65  (0.00% de los tokens de train)
  <name>             16  (0.00% de los tokens de train)
  <email>             3  (0.00% de los tokens de train)
  <ref_num>           1  (0.00% de los tokens de train)
  <user> 

In [18]:
# ¿Qué palabras se quedan sin vector? Las más frecuentes (en train) y una clasificación aproximada
oov_train = Counter({w: c for w, c in word_counts.items() if not is_marker(w) and not has_pre(w)})
oov_stop = sum(c for w, c in oov_train.items() if w in STOP_EN)
print(f"Tokens OOV que son stopwords: {oov_stop} de {sum(oov_train.values())} ({oov_stop / max(1, sum(oov_train.values())):.1%})\n")


def oov_category(w: str) -> str:
    if re.search(r"\d", w):
        return "contiene dígitos"
    if "'" in w:
        return "con apóstrofo (no resuelto por el fallback)"
    if "_" in w:
        return "con guion bajo"
    if not w.isascii():
        return "caracteres no ASCII"
    return "alfabético (jerga, producto, typo, nombre propio...)"


print(f"Top 30 OOV de train (de {len(oov_train)} tipos sin vector):")
for w, c in oov_train.most_common(30):
    print(f"  {w:<28} {c:>6}")

cat_types, cat_tokens = Counter(), Counter()
for w, c in oov_train.items():
    cat = oov_category(w)
    cat_types[cat] += 1
    cat_tokens[cat] += c
print("\nReparto de los OOV por categoría:")
print(pd.DataFrame({"tipos": cat_types, "tokens": cat_tokens}).fillna(0).astype(int).sort_values("tokens", ascending=False))

Tokens OOV que son stopwords: 0 de 4715 (0.0%)

Top 30 OOV de train (de 204 tipos sin vector):
  clickup                         451
  elasticsearch                   296
  airtable                        226
  19c                             197
  zapier                          194
  kubernetes                      192
  mongodb                         168
  rapidminer                      144
  datarobot                       125
  woocommerce                     104
  activecampaign                  102
  pytorch                         100
  sendgrid                         99
  mailchimp                        96
  laravel                          93
  ifttt                            73
  intellij                         70
  scikit                           69
  docusign                         68
  figma                            67
  tensorflow                       59
  gitlab                           54
  ccleaner                         52
  nod32                        

In [19]:
def build_embedding_matrix(vocab, get_vec, dim, seed=42):
    """Matriz [len(vocab), dim] alineada con los índices de Vocabulary.
    <UNK> -> ceros; marcadores y OOV -> vector aleatorio pequeño (se informa cuántos hay de cada tipo)."""
    rng = np.random.default_rng(seed)
    M = np.zeros((len(vocab), dim), dtype=np.float32)
    status = Counter()
    for idx in range(len(vocab)):
        tok = vocab.lookup_index(idx)
        if idx == vocab.unk_index:
            status["<UNK> (ceros)"] += 1
            continue
        vec = get_vec(tok)
        if vec is not None:
            M[idx] = vec
            status["con vector"] += 1
        else:
            M[idx] = rng.normal(0, 0.1, dim)
            status["marcador (aleatorio)" if is_marker(tok) else "OOV (aleatorio)"] += 1
    return M, status


def doc_embeddings(token_strings, vocab, M, drop_stopwords=True):
    """Un vector por ticket: media de los vectores de sus tokens (sin <UNK> y, opcionalmente, sin stopwords)."""
    out = np.zeros((len(token_strings), M.shape[1]), dtype=np.float32)
    for i, s in enumerate(token_strings):
        toks = [t for t in s.split() if not (drop_stopwords and t in STOP_EN)]
        ids = [j for j in (vocab.lookup_token(t) for t in toks) if j != vocab.unk_index]
        if ids:
            out[i] = M[ids].mean(axis=0)
    return out


M_pre, status_pre = build_embedding_matrix(vocab, get_pre_vec, EMB_DIM)
print("Matriz Word2Vec preentrenado:", M_pre.shape, dict(status_pre))

tokens_all = df["tokens"].fillna("").to_numpy()
D_train_pre = doc_embeddings(tokens_all[idx_train], vocab, M_pre)
D_test_pre = doc_embeddings(tokens_all[idx_test], vocab, M_pre)
print("Embeddings de ticket:", D_train_pre.shape, D_test_pre.shape)

np.save(OUTDIR / "emb_matrix_pretrained.npy", M_pre)
np.save(OUTDIR / "doc_emb_pretrained_train.npy", D_train_pre)
np.save(OUTDIR / "doc_emb_pretrained_test.npy", D_test_pre)

Matriz Word2Vec preentrenado: (4897, 300) {'<UNK> (ceros)': 1, 'con vector': 4741, 'marcador (aleatorio)': 6, 'OOV (aleatorio)': 149}
Embeddings de ticket: (22608, 300) (5652, 300)


## 9. Word2Vec propio: entrenamiento, comparación de cobertura y decisión

El enunciado pide analizar entrenar nuestro propio Word2Vec **si el número de palabras sin representar es grande**. Se decide con la tabla de cobertura de la sección 8, con este criterio (ajustable en `THRESH_*`):

- **% de tokens OOV en test (sin stopwords)** por encima de ~5 %, o
- **% de tipos OOV** por encima de ~15-20 % **y** que además incluyan palabras frecuentes del dominio.

Se entrenan dos variantes con los mismos hiperparámetros sobre `features/w2v_corpus_train.txt` (solo train):

| modelo | texto de entrenamiento |
|---|---|
| `w2v_all` | tickets (`subject` + `body`) + respuestas (`answer`) |
| `w2v_tickets` | solo tickets |

Mismo algoritmo que el modelo preentrenado, pero entrenado con el vocabulario del dominio. Parámetros: `sg=1` (skip-gram, mejor con corpus pequeños y palabras poco frecuentes), `vector_size=300` (igual que el preentrenado, para poder intercambiar las matrices), `window=5`, `min_count=2` (igual que `MIN_FREQ`), `epochs=20`, `seed=42`.

> **Lectura correcta de la cobertura de Word2Vec propio.** Al entrenar con el mismo texto que construye el vocabulario, la cobertura sobre *train* es casi 100 % **por construcción** (con `min_count` = `MIN_FREQ` es idéntica). La cifra que de verdad importa es la de **test**, que mide cuántas palabras nuevas aparecen que el modelo nunca vio.
>
> **Reproducibilidad.** Con `workers > 1` gensim no garantiza resultados idénticos entre ejecuciones aunque se fije `seed`. Para reproducir exactamente, usar `workers=1`.

In [ ]:
W2V_PARAMS = dict(vector_size=EMB_DIM, window=5, min_count=MIN_FREQ, sg=1, epochs=20, workers=4, seed=42)

w2v_all = Word2Vec(LineSentence(str(W2V_PATH)), **W2V_PARAMS)  # tickets + respuestas (solo train)
w2v_tickets = Word2Vec(w2v_ticket, **W2V_PARAMS)               # solo tickets (solo train)

for name, m in [("w2v_all", w2v_all), ("w2v_tickets", w2v_tickets)]:
    print(f"{name}: {len(m.wv.key_to_index):,} palabras, {m.wv.vector_size} dimensiones")

In [ ]:
has_w2v_all = lambda w: w in w2v_all.wv.key_to_index
has_w2v_tickets = lambda w: w in w2v_tickets.wv.key_to_index

rows = []
for emb_name, has_fn in [("preentrenado (Google News)", has_pre), ("w2v_all (propio)", has_w2v_all), ("w2v_tickets (propio)", has_w2v_tickets)]:
    for set_name, counter, mc, sk in [("train vocab", word_counts, MIN_FREQ, False),
                                      ("test", test_counts, 1, False),
                                      ("test sin stopwords", test_counts, 1, True)]:
        r = coverage_report(counter, has_fn, set_name, min_count=mc, skip_stop=sk)
        r["embedding"] = emb_name
        rows.append(r)

cov_all = pd.DataFrame(rows)[["embedding", "conjunto", "tipos", "% tipos OOV", "tokens", "% tokens OOV"]]
print(cov_all.to_string(index=False))

# Criterio de decisión
THRESH_TOKENS, THRESH_TYPES = 5.0, 15.0
g = cov_pre.iloc[3]  # fila: test sin stopwords, con el modelo preentrenado
need_own = g["% tokens OOV"] > THRESH_TOKENS or g["% tipos OOV"] > THRESH_TYPES
print(f"\nPreentrenado en test (sin stopwords): {g['% tipos OOV']}% de tipos y {g['% tokens OOV']}% de tokens sin vector")
print("-> Se recomienda usar Word2Vec propio." if need_own else
      "-> El preentrenado cubre suficiente; el Word2Vec propio es opcional (útil sobre todo para el vocabulario del dominio).")

In [ ]:
# Comprobación cualitativa: vecinos más cercanos de palabras frecuentes del dominio en cada espacio
# (most_similar exige que la palabra exista de verdad en cada espacio, sin los fallbacks de mayúsculas ni posesivos)
probe = [w for w, _ in word_counts.most_common(300)
         if w not in STOP_EN and not is_marker(w)
         and w in w2v_pre.key_to_index and has_w2v_all(w) and has_w2v_tickets(w)][:5]

for w in probe:
    print(f"\n{w}")
    print("  preentrenado:", [x for x, _ in w2v_pre.most_similar(w, topn=5)])
    print("  w2v_all    :", [x for x, _ in w2v_all.wv.most_similar(w, topn=5)])
    print("  w2v_tickets:", [x for x, _ in w2v_tickets.wv.most_similar(w, topn=5)])

In [ ]:
# Matriz alineada con Vocabulary para el Word2Vec propio (mismo formato que la preentrenada) + modelos guardados
get_w2v_vec = lambda w: w2v_all.wv[w] if w in w2v_all.wv.key_to_index else None

M_w2v, status_w2v = build_embedding_matrix(vocab, get_w2v_vec, EMB_DIM)
print("Matriz Word2Vec propio:", M_w2v.shape, dict(status_w2v))

D_train_w2v = doc_embeddings(tokens_all[idx_train], vocab, M_w2v)
D_test_w2v = doc_embeddings(tokens_all[idx_test], vocab, M_w2v)

np.save(OUTDIR / "emb_matrix_own.npy", M_w2v)
np.save(OUTDIR / "doc_emb_own_train.npy", D_train_w2v)
np.save(OUTDIR / "doc_emb_own_test.npy", D_test_w2v)
w2v_all.save(str(OUTDIR / "w2v_all.model"))
w2v_tickets.save(str(OUTDIR / "w2v_tickets.model"))
cov_all.to_csv(OUTDIR / "embedding_coverage.csv", index=False)  # tabla de cobertura para la memoria

## 10. Exportación estructurada (JSONL + metadatos)

Todo el texto procesado queda en un único fichero **`corpus.jsonl`** (un ticket por línea) junto con **`meta.json`**, de modo que cualquier tarea posterior (clasificación de `queue`/`priority`/`tags`, redes neuronales, etc.) pueda reutilizarlo sin repetir el preprocesado.

**Estructura de cada registro**

```json
{
  "id": 0,
  "split": "train",
  "language": "en",
  "labels": {"queue": "...", "priority": "...", "type": "...", "tags": ["...", "..."]},
  "text": {
    "subject": "...",
    "sentences": ["...", "..."],
    "tokens": ["dear", "customer", "..."],
    "token_ids": [12, 87, 5]
  },
  "aux": {"answer_tokens": ["thank", "you", "..."]}
}
```

- `id`: posición del ticket en `dataprocessed.csv` (coincide con `idx_train` / `idx_test` y con las filas de las matrices `.npz` / `.npy`).
- `labels`: etiquetas a predecir. **Son las únicas que contienen información de anotación.**
- `text.token_ids`: índices según `vocab.json` (`0`... según el orden de frecuencia; los tokens fuera de vocabulario valen el índice de `<UNK>`).
- `aux.answer_tokens`: tokens de la respuesta del agente. **No debe usarse como feature** (no existe cuando llega el ticket); se guarda solo por trazabilidad y para el corpus de Word2Vec.

**Ficheros generados**

| fichero | contenido |
|---|---|
| `dataprocessed/dataprocessed.csv` | tabla completa (texto original, oraciones, tokens, split) |
| `features/corpus.jsonl` | corpus estructurado (formato de reutilización principal) |
| `features/meta.json` | parámetros, tamaños, mapeos de etiquetas y lista de ficheros |
| `features/labels.json` | mapeo etiqueta → entero para `queue`, `priority` y `type` |
| `features/vocab.json` | `Vocabulary` serializado |
| `features/X_train.npz`, `X_test.npz`, `tfidf_vectorizer.joblib` | TF-IDF |
| `features/emb_matrix_{pretrained,own}.npy` | matrices de embeddings (Word2Vec preentrenado / propio) alineadas con `vocab.json` |
| `features/doc_emb_{pretrained,own}_{train,test}.npy` | un vector por ticket |
| `features/w2v_*.model`, `w2v_corpus_train.txt` | Word2Vec propio y su corpus |

In [ ]:
tag_cols = [c for c in df.columns if re.fullmatch(r"tag_?\d+", c)]  # tag_1 ... tag_N


def nn(v):
    """NaN -> None (JSON válido)."""
    return None if pd.isna(v) else v


JSONL_PATH = OUTDIR / "corpus.jsonl"
with open(JSONL_PATH, "w", encoding="utf-8") as fp:
    for i, row in enumerate(df.to_dict("records")):
        toks = row["tokens"].split() if isinstance(row["tokens"], str) else []
        ans = row["answer_tokens"].split() if isinstance(row["answer_tokens"], str) else []
        rec = {
            "id": i,
            "split": row["split"],
            "language": nn(row.get("language")),
            "labels": {
                "queue": nn(row.get("queue")),
                "priority": nn(row.get("priority")),
                "type": nn(row.get("type")),
                "tags": [t for t in (nn(row[c]) for c in tag_cols) if t is not None],
            },
            "text": {
                "subject": nn(row.get("subject")),
                "sentences": json.loads(row["sentences"]),
                "tokens": toks,
                "token_ids": [vocab.lookup_token(t) for t in toks],
            },
            "aux": {"answer_tokens": ans},
        }
        fp.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Mapeos etiqueta -> entero (orden alfabético, reproducible)
label_maps = {
    col: {v: i for i, v in enumerate(sorted(df[col].dropna().unique()))}
    for col in ["queue", "priority", "type"] if col in df.columns
}
with open(OUTDIR / "labels.json", "w", encoding="utf-8") as fp:
    json.dump(label_maps, fp, ensure_ascii=False, indent=2)

meta = {
    "dataset": "Tobi-Bueck/customer-support-tickets",
    "n_tickets": int(len(df)),
    "n_train": int((df["split"] == "train").sum()),
    "n_test": int((df["split"] == "test").sum()),
    "split": {"test_size": 0.2, "random_state": 42, "stratify": "queue"},
    "cleaning": "se eliminan body vacío y tickets en alemán (language == 'de')",
    "tokenization": "oraciones (saltos de línea y . ! ?) -> normalización (<url>, <email>, <num>, minúsculas) -> regex; sin lematizar ni quitar stopwords",
    "vocabulary": {"min_freq": MIN_FREQ, "size": len(vocab), "unk_token": "<UNK>", "unk_index": vocab.unk_index},
    "tfidf": {"shape_train": list(X_train.shape), "ngram_range": [1, 2], "min_df": 2, "max_df": 0.9, "stop_words": "spaCy EN"},
    "embeddings": {
        "pretrained": {"name": EMB_NAME, "dim": EMB_DIM},
        "word2vec_own": {k: v for k, v in W2V_PARAMS.items()},
    },
    "label_columns": ["queue", "priority", "type", "tags"],
    "tag_columns": tag_cols,
    "label_maps": label_maps,
    "feature_policy": "answer NO es feature de clasificación; solo se usa (train) como texto extra para Word2Vec",
}
with open(OUTDIR / "meta.json", "w", encoding="utf-8") as fp:
    json.dump(meta, fp, ensure_ascii=False, indent=2)

print(f"{JSONL_PATH} -> {sum(1 for _ in open(JSONL_PATH, encoding='utf-8'))} registros")

In [ ]:
# Comprobación: se puede reconstruir todo desde los ficheros, sin el notebook
with open(OUTDIR / "vocab.json", encoding="utf-8") as fp:
    vocab2 = Vocabulary.from_serializable(json.load(fp))
assert len(vocab2) == len(vocab)

with open(JSONL_PATH, encoding="utf-8") as fp:
    records = [json.loads(line) for line in fp]
assert len(records) == len(df)
assert all(r["id"] == i for i, r in enumerate(records))
assert all(len(r["text"]["tokens"]) == len(r["text"]["token_ids"]) for r in records)
assert all(vocab2.lookup_token(t) == j for t, j in zip(records[0]["text"]["tokens"], records[0]["text"]["token_ids"]))
assert [r["id"] for r in records if r["split"] == "train"] == list(idx_train)  # mismo orden que X_train / doc_emb_*_train

r = records[0]
print(json.dumps({**r, "text": {**r["text"], "sentences": r["text"]["sentences"][:2],
                                "tokens": r["text"]["tokens"][:8], "token_ids": r["text"]["token_ids"][:8]},
                  "aux": {"answer_tokens": r["aux"]["answer_tokens"][:8]}}, ensure_ascii=False, indent=2))

# Inventario de ficheros generados
for p in sorted(OUTDIR.glob("*")) + [Path(PROCESSED_PATH)]:
    print(f"{p.stat().st_size / 1e6:8.2f} MB  {p}")